In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, log_loss, accuracy_score
from sklearn.preprocessing import  OneHotEncoder, LabelEncoder
from sklearn.compose import make_column_transformer, make_column_selector
from tqdm import tqdm

In [2]:
df = pd.read_csv("Housing.csv")
display(df.head())
print("Shape:",df.shape)
print("Data Type:",df.dtypes)
print("Missing values count:",df.isnull().sum())



,price,lotsize,bedrooms,bathrms,stories,driveway,recroom,fullbase,gashw,airco,garagepl,prefarea
0,42000.0,5850,3,1,2,yes,no,yes,no,no,1,no
1,38500.0,4000,2,1,1,yes,no,no,no,no,0,no
2,49500.0,3060,3,1,1,yes,no,no,no,no,0,no
3,60500.0,6650,3,1,2,yes,yes,no,no,no,0,no
4,61000.0,6360,2,1,1,yes,no,no,no,no,0,no


Shape: (546, 12)
Data Type: price       float64
lotsize       int64
bedrooms      int64
bathrms       int64
stories       int64
driveway        str
recroom         str
fullbase        str
gashw           str
airco           str
garagepl      int64
prefarea        str
dtype: object
Missing values count: price       0
lotsize     0
bedrooms    0
bathrms     0
stories     0
driveway    0
recroom     0
fullbase    0
gashw       0
airco       0
garagepl    0
prefarea    0
dtype: int64


In [3]:
#Set price as the target and use the remaining columns as features
X=df.drop(columns=['price'])
y=df['price']

In [4]:
#Split the dataset into 70% training and 30% testing, using random_state=25. Use the
#same split throughout the assignment.
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.30,random_state=25)

In [5]:
#Apply OneHotEncoder to categorical features with drop='first' and sparse_output=False.
#Keep numerical features unchanged.
#Fit the preprocessing transformer on the training data only, then transform both training
#and test data.


ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
trns = make_column_transformer(
    (ohe, make_column_selector(dtype_include=['object', 'category'])),
    remainder='passthrough', 
    verbose_feature_names_out=False
)
trns.set_output(transform='pandas')

X_train_trans_df = trns.fit_transform(X_train)
X_test_trans_df = trns.transform(X_test)

In [6]:
import itertools
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

In [7]:


# Define parameter grid
lr_values = [0.01, 0.05, 0.1]
n_est_values = [50, 100]
max_depth_values = [3, 5]


xgb_results = []

for lr, n_est, depth in itertools.product(lr_values, n_est_values, max_depth_values):
    # Initialize and train model
    model = XGBRegressor(
        learning_rate=lr,
        n_estimators=n_est,
        max_depth=depth,
        random_state=25
       
    )
    model.fit(X_train_trans_df, y_train)
    
    # Predict using y_pred
    y_pred = model.predict(X_test_trans_df)
    
    # Calculate R² score
    r2 = r2_score(y_test, y_pred)
    
    xgb_results.append({
        'Learning Rate': lr,
        'Number of Estimators': n_est,
        'Maximum Depth': depth,
        'R² Score': r2
    })

# Create Results DataFrame
xgb_results_df = pd.DataFrame(xgb_results)

# Sort results from highest to lowest R²
xgb_results_df = xgb_results_df.sort_values(by='R² Score', ascending=False).reset_index(drop=True)

print("--- XGBoost Results (Sorted) ---")
display(xgb_results_df)

# Identify highest-scoring configuration
best_xgb = xgb_results_df.iloc[0]
print(f"\nBest XGBoost Configuration:\n{best_xgb}")

--- XGBoost Results (Sorted) ---


,Learning Rate,Number of Estimators,Maximum Depth,R² Score
0,0.05,100,3,0.619739
1,0.10,50,3,0.616997
2,0.10,100,3,0.602814
3,0.05,50,3,0.596801
4,0.05,100,5,0.588742
5,0.05,50,5,0.582350
6,0.10,50,5,0.570771
7,0.10,100,5,0.560056
8,0.01,100,5,0.488779
9,0.01,100,3,0.450938



Best XGBoost Configuration:
Learning Rate             0.050000
Number of Estimators    100.000000
Maximum Depth             3.000000
R² Score                  0.619739
Name: 0, dtype: float64


In [8]:
from lightgbm import LGBMRegressor

In [9]:

 

# Define parameter grid
lr_values = [0.01, 0.05, 0.1]
n_est_values = [50, 100,200]
max_depth_values = [3, 5,-1]


lg_results = []

for lr, n_est, depth in itertools.product(lr_values, n_est_values, max_depth_values):
    # Initialize and train model
    model = LGBMRegressor(
        learning_rate=lr,
        n_estimators=n_est,
        max_depth=depth,
        random_state=25
       
    )
    model.fit(X_train_trans_df, y_train)
    
    # Predict using y_pred
    y_pred = model.predict(X_test_trans_df)
    
    # Calculate R² score
    r2 = r2_score(y_test, y_pred)
    
    lg_results.append({
        'Learning Rate': lr,
        'Number of Estimators': n_est,
        'Maximum Depth': depth,
        'R² Score': r2
    })

# Create Results DataFrame
lg_results_df = pd.DataFrame(lg_results)

# Sort results from highest to lowest R²
lg_results_df = lg_results_df.sort_values(by='R² Score', ascending=False).reset_index(drop=True)

print("--- LGM Results (Sorted) ---")
display(lg_results_df)

# Identify highest-scoring configuration
best_lgm = lg_results_df.iloc[0]
print(f"\nBest XGBoost Configuration:\n{best_lgm}")



[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000729 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 123
[LightGBM] [Info] Number of data points in the train set: 382, number of used features: 10
[LightGBM] [Info] Start training from score 66491.209424
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

,Learning Rate,Number of Estimators,Maximum Depth,R² Score
0,0.05,100,-1,0.641652
1,0.10,50,-1,0.640175
2,0.10,50,3,0.635631
3,0.05,100,5,0.634652
4,0.05,100,3,0.633551
5,0.10,50,5,0.626121
6,0.05,50,-1,0.621275
7,0.05,200,-1,0.620178
8,0.10,100,3,0.620093
9,0.05,50,5,0.618426



Best XGBoost Configuration:
Learning Rate             0.050000
Number of Estimators    100.000000
Maximum Depth            -1.000000
R² Score                  0.641652
Name: 0, dtype: float64


In [10]:
from catboost import CatBoostRegressor


In [11]:


# Define parameter lists matching your assignment
lr_values = [0.01, 0.05, 0.1]
n_est_values = [50, 100]
max_depth_values = [3, 5]


cat_results_ohe = []

# Using nested for loops with tqdm matching your teacher's style
for lr in tqdm(lr_values, desc="Learning Rates"):
    for n_est in n_est_values:
        for depth in max_depth_values:
            # Initialize and train model (using iterations for n_estimators in CatBoost)
            model = CatBoostRegressor(
                learning_rate=lr,
                iterations=n_est,
                depth=depth,
                random_seed=25,
                verbose=0
            )
            model.fit(X_train_trans_df, y_train)
            y_pred = model.predict(X_test_trans_df)
            r2 = r2_score(y_test, y_pred)
            
            cat_results_ohe.append({
                'Learning Rate': lr,
                'Number of Estimators': n_est,
                'Maximum Depth': depth,
                'R² Score': r2
            })

# Create Results DataFrame and sort descending
cat_ohe_results_df = pd.DataFrame(cat_results_ohe)
cat_ohe_results_df = cat_ohe_results_df.sort_values(by='R² Score', ascending=False).reset_index(drop=True)

print("--- CatBoost with OHE Results (Sorted) ---")
display(cat_ohe_results_df)

best_cat_ohe = cat_ohe_results_df.iloc[0]
print(f"\nBest CatBoost (OHE) Configuration:\n{best_cat_ohe}")

Learning Rates: 100%|██████████| 3/3 [00:03<00:00,  1.10s/it]

--- CatBoost with OHE Results (Sorted) ---


,Learning Rate,Number of Estimators,Maximum Depth,R² Score
0,0.10,100,3,0.632009
1,0.10,50,5,0.628983
2,0.10,100,5,0.627189
3,0.05,100,5,0.624611
4,0.10,50,3,0.617346
5,0.05,100,3,0.612241
6,0.05,50,5,0.585247
7,0.05,50,3,0.579198
8,0.01,100,5,0.439275
9,0.01,100,3,0.414968



Best CatBoost (OHE) Configuration:
Learning Rate             0.100000
Number of Estimators    100.000000
Maximum Depth             3.000000
R² Score                  0.632009
Name: 0, dtype: float64


In [12]:

cat_feature_names = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Ensure categorical columns are formatted as string type for CatBoost native support
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()
for col in cat_feature_names:
    X_train_raw[col] = X_train_raw[col].astype(str)
    X_test_raw[col] = X_test_raw[col].astype(str)

# 17. Train baseline CatBoostRegressor with default parameters and cat_features
baseline_cat = CatBoostRegressor(
    random_seed=25,
    cat_features=cat_feature_names,
    verbose=0
)
baseline_cat.fit(X_train_raw, y_train)

# 18. Predict test-set house prices and calculate baseline R² score
y_pred = baseline_cat.predict(X_test_raw)
baseline_r2 = r2_score(y_test, y_pred)
print(f"Baseline CatBoost (Native Categorical) Test R² Score: {baseline_r2:.5f}\n")

# 19 & 20. Repeat parameter combinations using original features and cat_features (new results list)
cat_results_native = []

for lr, n_est, depth in itertools.product(lr_values, n_est_values, max_depth_values):
    model = CatBoostRegressor(
        learning_rate=lr,
        iterations=n_est,
        depth=depth,
        random_seed=25,
        cat_features=cat_feature_names,
        verbose=0
    )
    model.fit(X_train_raw, y_train)
    
    y_pred = model.predict(X_test_raw)
    r2 = r2_score(y_test, y_pred)
    
    cat_results_native.append({
        'Learning Rate': lr,
        'Number of Estimators': n_est,
        'Maximum Depth': depth,
        'R² Score': r2
    })

cat_native_results_df = pd.DataFrame(cat_results_native)
cat_native_results_df = cat_native_results_df.sort_values(by='R² Score', ascending=False).reset_index(drop=True)

print("--- CatBoost without OHE (Native) Results (Sorted) ---")
display(cat_native_results_df)

best_cat_native = cat_native_results_df.iloc[0]
print(f"\nBest CatBoost (Native Categorical) Configuration:\n{best_cat_native}")

C:\Users\HP\AppData\Local\Temp\ipykernel_4100\893789786.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_feature_names = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Baseline CatBoost (Native Categorical) Test R² Score: 0.61584

--- CatBoost without OHE (Native) Results (Sorted) ---


,Learning Rate,Number of Estimators,Maximum Depth,R² Score
0,0.10,100,3,0.629069
1,0.05,100,5,0.627412
2,0.10,100,5,0.621193
3,0.10,50,3,0.620838
4,0.10,50,5,0.617123
5,0.05,100,3,0.614771
6,0.05,50,5,0.590492
7,0.05,50,3,0.576047
8,0.01,100,5,0.437304
9,0.01,100,3,0.416712



Best CatBoost (Native Categorical) Configuration:
Learning Rate             0.100000
Number of Estimators    100.000000
Maximum Depth             3.000000
R² Score                  0.629069
Name: 0, dtype: float64


In [13]:
# Create summary table DataFrame combining the best configuration from each experiment



summary_data = [
    {
        'Experiment': 'XGBoost',
        'Learning Rate': best_xgb['Learning Rate'],
        'Number of Estimators': int(best_xgb['Number of Estimators']),
        'Maximum Depth': int(best_xgb['Maximum Depth']),
        'Test R²': round(best_xgb['R² Score'], 5)
    },
    {
        'Experiment': 'LightGBM',
        'Learning Rate':best_lgm['Learning Rate'],
        'Number of Estimators': int(best_lgm['Number of Estimators']),
        'Maximum Depth': int(best_lgm['Maximum Depth']),
        'Test R²': round(best_lgm['R² Score'], 5)
    },
    {
        'Experiment': 'CatBoost with One-Hot Encoding',
        'Learning Rate': best_cat_ohe['Learning Rate'],
        'Number of Estimators': int(best_cat_ohe['Number of Estimators']),
        'Maximum Depth': int(best_cat_ohe['Maximum Depth']),
        'Test R²': round(best_cat_ohe['R² Score'], 5)
    },
    {
        'Experiment': 'CatBoost without One-Hot Encoding',
        'Learning Rate': best_cat_native['Learning Rate'],
        'Number of Estimators': int(best_cat_native['Number of Estimators']),
        'Maximum Depth': int(best_cat_native['Maximum Depth']),
        'Test R²': round(best_cat_native['R² Score'], 5)
    }
]

summary_df = pd.DataFrame(summary_data)
print("--- Summary of Best Configurations ---")
display(summary_df)

--- Summary of Best Configurations ---


,Experiment,Learning Rate,Number of Estimators,Maximum Depth,Test R²
0,XGBoost,0.05,100,3,0.61974
1,LightGBM,0.05,100,-1,0.64165
2,CatBoost with One-Hot Encoding,0.10,100,3,0.63201
3,CatBoost without One-Hot Encoding,0.10,100,3,0.62907
